# Панельная регрессия с устранением мультиколлинеарности
В этом ноутбуке мы импортируем набор данных `cleaned_df.csv` и строим панельные модели (Fixed Effects, Random Effects).
Признаки автоматически проверяются на мультиколлинеарность (с помощью VIF > 10), после чего сильно коррелирующие регрессоры итеративно удаляются из выборки для стабилизации оценок.

In [64]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from linearmodels.panel import compare
import warnings

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('cleaned_df.csv')

In [3]:
region_col = df.columns[0]
year_col = df.columns[1]
panel_df = df.set_index([region_col, year_col]).sort_index()

In [4]:
panel_df['ln_ВРП_на_душу'] = np.log(panel_df['Валовой региональный продукт на душу населения (в текущих основных ценах), рублей'])
panel_df['ln_Доходы'] = np.log(panel_df['Среднедушевые денежные доходы населения, рублей'])
panel_df['ln_Экспорт'] = np.log(panel_df['Объем экспорта, млн долл.'])
panel_df['lag_ln_Доходы'] = panel_df.groupby(level=0)['ln_Доходы'].shift(1)
panel_df['lag_Ввод_жилья'] = panel_df.groupby(level=0)['Введено общей площади жилых помещений (с учетом жилых домов на участках для ведения садоводства), тыс. кв. м'].shift(1)

In [69]:


target_var = 'Вторичный рынок. Индекс цен на рынке жилья (все типы квартир)'

all_cols = [c for c in panel_df.columns if c not in [target_var, 'region', 'year']]

log_candidates = [
    c for c in all_cols
    if 'индекс' not in c.lower() and '%' not in c and 'коэффициент' not in c.lower()
    and 'темп' not in c.lower() and 'доля' not in c.lower() and 'уровень' not in c.lower()
]

for col in log_candidates:
    if (panel_df[col] > 0).all():
        panel_df[f'ln_{col}'] = np.log(panel_df[col])
        all_cols.remove(col)  
        all_cols.append(f'ln_{col}') 

# ТОП-10
y_temp = panel_df[target_var]
X_temp = panel_df[all_cols]

corrs = X_temp.corrwith(y_temp).abs()  
top_10_features = corrs.nlargest(10).index.tolist()

print("Топ 10 самых значимых регрессоров (по корреляции с таргетом):")
for i, f in enumerate(top_10_features, 1):
    print(f"{i}. {f} (corr: {corrs[f]:.3f})")

panel_df_clean = panel_df[[target_var] + top_10_features].dropna()

y = panel_df_clean[target_var]
X = panel_df_clean[top_10_features]

# X = sm.add_constant(X)


Топ 10 самых значимых регрессоров (по корреляции с таргетом):
1. Среднемесячная номинальная начисленная заработная плата, в % к СППГ (corr: 0.405)
2. Реальная начисленная заработная плата, в % к СППГ (corr: 0.350)
3. Реальные денежные доходы населения, в % к СППГ (corr: 0.287)
4. Коэффициент естественного прироста населения (на 1000 населения), человек (corr: 0.245)
5. Коэффициент смертности (на 1000 населения), человек (corr: 0.240)
6. Индекс оборота розничной торговли, в % к СППГ (corr: 0.239)
7. Индекс физического объема оборота розничной торговли непродовольственными товарами, в % к СППГ (corr: 0.191)
8. Индекс физического объема оборота розничной пищевыми продуктами, включая напитки, и табачными изделиями, в % к СППГ (corr: 0.188)
9. Реальные располагаемые денежные доходы, к соответствующему периоду предыдущего года, в % (corr: 0.182)
10. Коэффициент рождаемости (на 1000 населения), человек (corr: 0.161)


### Проверка и устранение мультиколлинеарности
Итеративно удаляем признаки с VIF > 10. 
Чем выше VIF, тем сильнее признак линейно зависит от комбинации остальных предикторов, что ведет к смещенным стандартным ошибкам.

In [70]:
def calculate_vif(X_df):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_df.columns
    vif_data["VIF"] = [variance_inflation_factor(X_df.values, i) for i in range(X_df.shape[1])]
    return vif_data.sort_values(by='VIF', ascending=False)

vif_threshold = 60000000.0 
X_selected = X.copy()

while True:
    vif_df = calculate_vif(X_selected)
    vif_df_no_const = vif_df[vif_df['feature'] != 'const']
    if vif_df_no_const.empty:
        break
    
    max_vif_feature = vif_df_no_const.iloc[0]['feature']
    max_vif_value = vif_df_no_const.iloc[0]['VIF']
    
    if max_vif_value > vif_threshold:
        print(f"Удаляем признак '{max_vif_feature}' с VIF = {max_vif_value:.2f}")
        X_selected = X_selected.drop(columns=[max_vif_feature])
    else:
        break

display(calculate_vif(X_selected))

,feature,VIF
5,"Индекс оборота розничной торговли, в % к СППГ",17245.319869
7,Индекс физического объема оборота розничной пи...,5868.122836
6,Индекс физического объема оборота розничной то...,2944.621559
0,Среднемесячная номинальная начисленная заработ...,2069.366574
2,"Реальные денежные доходы населения, в % к СППГ",1860.634215
8,"Реальные располагаемые денежные доходы, к соот...",1668.860120
1,"Реальная начисленная заработная плата, в % к СППГ",1303.190963
4,"Коэффициент смертности (на 1000 населения), че...",355.353288
9,"Коэффициент рождаемости (на 1000 населения), ч...",192.304579
3,Коэффициент естественного прироста населения (...,17.108766


In [71]:
fe_model = PanelOLS(y, X_selected, entity_effects=True) 
fe_res = fe_model.fit(cov_type='clustered', cluster_entity=True)
print("============= Модель с Фиксированными (Регион и Время) Эффектами (FE) =============")
print(fe_res)

============= Модель с Фиксированными (Регион и Время) Эффектами (FE) =============
                                                PanelOLS Estimation Summary                                                
Dep. Variable:     Вторичный рынок. Индекс цен на рынке жилья (все типы квартир)   R-squared:                        0.3411
Estimator:                                                              PanelOLS   R-squared (Between):              0.9511
No. Observations:                                                            195   R-squared (Within):               0.3411
Date:                                                           Sun, Mar 15 2026   R-squared (Overall):              0.9482
Time:                                                                   00:26:21   Log-likelihood                   -625.92
Cov. Estimator:                                                        Clustered                                           
                                                

In [72]:
re_model = RandomEffects(y, X_selected)
re_res = re_model.fit()
print("\n============= Модель со Случайными Эффектами (RE) =============")
print(re_res)


============= Модель со Случайными Эффектами (RE) =============
                                              RandomEffects Estimation Summary                                             
Dep. Variable:     Вторичный рынок. Индекс цен на рынке жилья (все типы квартир)   R-squared:                        0.9961
Estimator:                                                         RandomEffects   R-squared (Between):              0.9994
No. Observations:                                                            195   R-squared (Within):               0.2937
Date:                                                           Sun, Mar 15 2026   R-squared (Overall):              0.9961
Time:                                                                   00:26:48   Log-likelihood                   -647.67
Cov. Estimator:                                                       Unadjusted                                           
                                                                   

In [73]:
pooled_model = PooledOLS(y, X_selected)
pooled_res = pooled_model.fit()
print("\n============= Сравнение Моделей =============")
print(compare({'Pooled OLS': pooled_res, 'Fixed Effects (2-way)': fe_res, 'Random Effects': re_res}))



============= Сравнение Моделей =============
                                                                                                                                                       Model Comparison                                                                                                                                                      
                                                                                                                                                                               Pooled OLS                                             Fixed Effects (2-way)                                                    Random Effects
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------